# Detector de huevos y protozoos — entrenamiento en Google Colab

Este cuaderno entrena un **YOLO11** y lo deja listo para probar con fotos.

Datos que junta, en este orden:

| Fuente | Qué aporta | Tamaño |
|---|---|---|
| [Chula-ParasiteEgg-11](https://zenodo.org/records/15866298) | 11 huevos de helmintos, ~1.000 fotos por clase | zip 9 GB |
| [Microscopio USB, Chula](https://zenodo.org/records/10437395) | Las mismas 4 clases (Ascaris, H. diminuta, Fasciolopsis, Taenia), fotos baratas a 10×. Solo los cuadros originales, no los aumentos ya horneados | 94 MB |
| [ParasitoBank](https://data.mendeley.com/datasets/725hpzpwzf/1) (Cauca, Colombia) | Protozoos de materia fecal fresca tomados con celular: Giardia, Blastocystis, Chilomastix, Endolimax, E. coli, E. hartmanni, Iodamoeba. Los helmintos que ya existen se fusionan, no se duplican como clase nueva | lo descargas tú, una vez |
| Tu carpeta `mis_anotaciones` | Fotos tuyas ya marcadas en formato YOLO | opcional |

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*. Ejecuta las celdas en orden.

| Paso | Tiempo aprox. |
|---|---|
| Descargar Chula (9 GB) y el set USB | 15–25 min |
| Entrenar (25 épocas, GPU T4) | 2–4 h |

El entrenamiento se guarda en Google Drive. Si Colab se corta, reanúdalo en el paso 5b.

No se mezcla sedimento urinario: células, cilindros y cristales son otra tarea y empeoran este detector.

Clases con menos de 40 ejemplos (Strongyloides, complejo Entamoeba) se dejan fuera: con tan pocas fotos el modelo las inventa.

> Herramienta educativa. No sustituye el diagnóstico de laboratorio. Cita a Palasuwan et al. (Chula, ICIP 2022) y a Muñoz Galindez et al. (ParasitoBank, Data in Brief 2025) si publicas algo.


## 1. Comprobar GPU e instalar librerías

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics gradio

## 2. Conectar Google Drive (para guardar el modelo)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROYECTO = '/content/drive/MyDrive/detector_parasitos'
!mkdir -p {PROYECTO}

## 3. Guardar los scripts del prototipo

In [ ]:
%%writefile convertir_coco_a_yolo.py
"""
Convierte las anotaciones COCO (labels.json) de Chula-ParasiteEgg-11 al formato YOLO.

Uso:
    python convertir_coco_a_yolo.py --json ruta/labels.json --imagenes ruta/carpeta_imagenes \
        --salida dataset_yolo --val 0.15

Busca las imágenes de forma recursiva por nombre de archivo, así que no importa
la estructura exacta de carpetas dentro del zip.
"""
import argparse
import json
import random
import shutil
from collections import defaultdict
from pathlib import Path

# Nombres en español (el orden sigue los category_id originales 0..10)
NOMBRES_ES = {
    "Ascaris lumbricoides": "Ascaris lumbricoides",
    "Capillaria philippinensis": "Capillaria philippinensis",
    "Enterobius vermicularis": "Enterobius vermicularis",
    "Fasciolopsis buski": "Fasciolopsis buski",
    "Hookworm egg": "Uncinaria (huevo)",
    "Hymenolepis diminuta": "Hymenolepis diminuta",
    "Hymenolepis nana": "Hymenolepis nana",
    "Opisthorchis viverrine": "Opisthorchis viverrini",
    "Paragonimus spp": "Paragonimus spp.",
    "Taenia spp. egg": "Taenia spp. (huevo)",
    "Trichuris trichiura": "Trichuris trichiura",
}

EXTENSIONES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def indexar_imagenes(carpeta: Path) -> dict:
    indice = {}
    for p in carpeta.rglob("*"):
        if p.suffix.lower() in EXTENSIONES:
            indice[p.name] = p
    return indice


def convertir(json_path: Path, carpeta_imgs: Path, salida: Path, frac_val: float, semilla: int = 42,
              copiar: bool = False):
    datos = json.loads(json_path.read_text())

    categorias = sorted(datos["categories"], key=lambda c: c["id"])
    id_a_clase = {c["id"]: i for i, c in enumerate(categorias)}
    nombres = [NOMBRES_ES.get(c["name"], c["name"]) for c in categorias]

    anots_por_imagen = defaultdict(list)
    for a in datos["annotations"]:
        anots_por_imagen[a["image_id"]].append(a)

    indice = indexar_imagenes(carpeta_imgs)
    imagenes = list(datos["images"])
    random.Random(semilla).shuffle(imagenes)
    n_val = int(len(imagenes) * frac_val)

    faltantes = 0
    conteo = {"train": 0, "val": 0}
    for i, img in enumerate(imagenes):
        split = "val" if i < n_val else "train"
        nombre = Path(img["file_name"]).name
        origen = indice.get(nombre)
        if origen is None:
            faltantes += 1
            continue

        dir_img = salida / "images" / split
        dir_lbl = salida / "labels" / split
        dir_img.mkdir(parents=True, exist_ok=True)
        dir_lbl.mkdir(parents=True, exist_ok=True)

        destino = dir_img / nombre
        if not destino.exists():
            if copiar:
                shutil.copy2(origen, destino)
            else:
                destino.symlink_to(origen.resolve())

        W, H = img["width"], img["height"]
        lineas = []
        for a in anots_por_imagen.get(img["id"], []):
            x, y, w, h = a["bbox"]  # COCO: esquina superior izquierda + ancho/alto en píxeles
            if w <= 0 or h <= 0:
                continue
            cx = min(max((x + w / 2) / W, 0), 1)
            cy = min(max((y + h / 2) / H, 0), 1)
            lineas.append(f"{id_a_clase[a['category_id']]} {cx:.6f} {cy:.6f} {min(w / W, 1):.6f} {min(h / H, 1):.6f}")
        (dir_lbl / (Path(nombre).stem + ".txt")).write_text("\n".join(lineas))
        conteo[split] += 1

    yaml = [f"path: {salida.resolve()}", "train: images/train", "val: images/val", "names:"]
    yaml += [f"  {i}: \"{n}\"" for i, n in enumerate(nombres)]
    (salida / "data.yaml").write_text("\n".join(yaml) + "\n")

    print(f"Clases ({len(nombres)}): {nombres}")
    print(f"Imágenes -> entrenamiento: {conteo['train']}, validación: {conteo['val']}, no encontradas: {faltantes}")
    print(f"Archivo de configuración: {salida / 'data.yaml'}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--json", required=True, type=Path)
    ap.add_argument("--imagenes", required=True, type=Path)
    ap.add_argument("--salida", default=Path("dataset_yolo"), type=Path)
    ap.add_argument("--val", default=0.15, type=float, help="Fracción para validación (0.15 = 15%%)")
    ap.add_argument("--copiar", action="store_true", help="Copiar imágenes en vez de crear enlaces")
    args = ap.parse_args()
    convertir(args.json, args.imagenes, args.salida, args.val, copiar=args.copiar)


In [ ]:
%%writefile app.py
"""
App web para detectar huevos de parásitos en fotos de microscopio (prototipo).

Uso:
    python app.py --modelo best.pt            # abre en http://127.0.0.1:7860
    python app.py --modelo best.pt --compartir  # genera un enlace público temporal (útil en Colab o celular)

AVISO: herramienta educativa / de apoyo. No sustituye la revisión de un profesional.
"""
import argparse

import gradio as gr
import numpy as np
from PIL import Image
from ultralytics import YOLO

AVISO = ("Prototipo educativo. Reconoce los 11 huevos de Chula y, si sumaste ParasitoBank, "
         "protozoos de materia fecal (Giardia, Blastocystis, amebas comensales, Chilomastix, Endolimax, Iodamoeba). "
         "No es un resultado de laboratorio: confirme al microscopio.")


def diametro_campo_px(img: np.ndarray) -> float | None:
    """Estima el diámetro (en píxeles) del círculo iluminado en fotos tomadas por el ocular."""
    gris = img.mean(axis=2) if img.ndim == 3 else img
    umbral = (gris.max() + gris.min()) / 2
    mascara = gris > umbral
    if mascara.mean() > 0.95 or mascara.mean() < 0.05:
        return None  # la foto no muestra un círculo oscuro alrededor
    filas = np.where(mascara.any(axis=1))[0]
    cols = np.where(mascara.any(axis=0))[0]
    return float(max(filas[-1] - filas[0], cols[-1] - cols[0]))


def construir_app(ruta_modelo: str) -> gr.Blocks:
    modelo = YOLO(ruta_modelo)

    def analizar(imagen, confianza, diam_campo_um):
        if imagen is None:
            return None, [], "Sube una imagen."
        arr = np.array(imagen.convert("RGB"))
        res = modelo.predict(arr, conf=confianza, verbose=False)[0]
        anotada = Image.fromarray(res.plot()[:, :, ::-1])

        um_por_px = None
        nota_escala = "Tamaño no calculado (indica el diámetro del campo para estimarlo)."
        if diam_campo_um and diam_campo_um > 0:
            d_px = diametro_campo_px(arr)
            if d_px:
                um_por_px = diam_campo_um / d_px
                nota_escala = f"Escala estimada: {um_por_px:.3f} µm/píxel (campo de {d_px:.0f} px)."
            else:
                nota_escala = "No se detectó el círculo del campo; tamaño no calculado."

        filas = []
        for caja in res.boxes:
            x1, y1, x2, y2 = caja.xyxy[0].tolist()
            clase = res.names[int(caja.cls)]
            tam = (f"{(x2 - x1) * um_por_px:.0f} × {(y2 - y1) * um_por_px:.0f}" if um_por_px else "—")
            filas.append([clase, f"{float(caja.conf):.0%}", tam])

        resumen = (f"Se detectaron {len(filas)} estructura(s)." if filas
                   else "No se detectó ningún huevo conocido con esta confianza.")
        return anotada, filas, f"{resumen}\n{nota_escala}\n\n{AVISO}"

    with gr.Blocks(title="Detector de huevos de parásitos") as app:
        gr.Markdown("# 🔬 Detector de huevos de parásitos (prototipo)\n" + AVISO)
        with gr.Row():
            with gr.Column():
                entrada = gr.Image(type="pil", label="Foto del microscopio", sources=["upload", "webcam"])
                confianza = gr.Slider(0.05, 0.95, value=0.25, step=0.05, label="Confianza mínima")
                diam = gr.Number(value=0, label="Diámetro del campo en µm (opcional; ~450 µm a 40x, ~1800 µm a 10x)")
                boton = gr.Button("Analizar", variant="primary")
            with gr.Column():
                salida_img = gr.Image(label="Resultado")
                tabla = gr.Dataframe(headers=["Estructura", "Confianza", "Tamaño aprox. (µm)"], label="Detecciones")
                texto = gr.Textbox(label="Notas", lines=4)
        boton.click(analizar, [entrada, confianza, diam], [salida_img, tabla, texto])
    return app


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--modelo", default="best.pt")
    ap.add_argument("--compartir", action="store_true")
    args = ap.parse_args()
    construir_app(args.modelo).launch(share=args.compartir)


## 4. Descargar y preparar Chula-ParasiteEgg-11
Fuente: [Zenodo](https://zenodo.org/records/15866298). Cita a los autores (Palasuwan et al., ICIP 2022) si publicas algo.

In [ ]:
%cd /content
!wget -q --show-progress -c -O chula.zip "https://zenodo.org/records/15866298/files/Chula-ParasiteEgg-11.zip?download=1"
!unzip -q -o chula.zip -d chula && rm chula.zip
import glob
jsons = [p for p in glob.glob('/content/chula/**/*.json', recursive=True)]
print('Anotaciones encontradas:', jsons)
LABELS = jsons[0]

In [ ]:
!python convertir_coco_a_yolo.py --json "$LABELS" --imagenes /content/chula --salida /content/dataset_yolo --val 0.15

## 4b. Sumar más datos al mismo `dataset_yolo`

Corre esto **después** del paso 4 y **antes** de entrenar.

- El set USB se baja solo.
- ParasitoBank no tiene enlace anónimo estable. Descárgalo una vez desde [Mendeley](https://data.mendeley.com/datasets/725hpzpwzf/1) (*Download All*) y déjalo en Drive como `detector_parasitos/ParasitoBank.zip`. Si no está, el entrenamiento sigue solo con Chula + USB.
- Tus propias marcas: en Drive, `detector_parasitos/mis_anotaciones/` con `images/`, `labels/` (txt YOLO) y `classes.txt` (un nombre por línea, igual que en el atlas: `Giardia duodenalis`, `Ascaris lumbricoides`…).


In [ ]:
import random
import shutil
import zipfile
from pathlib import Path

import yaml
from PIL import Image

SALIDA = Path("/content/dataset_yolo")
YAML = SALIDA / "data.yaml"
RANDOM = random.Random(42)
MIN_NUEVA = 40  # una clase que Chula no trae, con menos fotos, se omite

SINONIMOS = [
    ("entamoeba complex", None),
    ("complejo entamoeba", None),
    ("entamoeba histolytica", None),
    ("ascaris", "Ascaris lumbricoides"),
    ("trichuris", "Trichuris trichiura"),
    ("taenia", "Taenia spp. (huevo)"),
    ("hymenolepis diminuta", "Hymenolepis diminuta"),
    ("hymenolepis nana", "Hymenolepis nana"),
    ("enterobius", "Enterobius vermicularis"),
    ("capillaria", "Capillaria philippinensis"),
    ("fascilopsis", "Fasciolopsis buski"),
    ("fasciolopsis", "Fasciolopsis buski"),
    ("large egg", "Fasciolopsis buski"),
    ("hookworm", "Uncinaria (huevo)"),
    ("uncinaria", "Uncinaria (huevo)"),
    ("opisthorchis", "Opisthorchis viverrini"),
    ("paragonimus", "Paragonimus spp."),
    ("giardia", "Giardia duodenalis"),
    ("blastocystis", "Blastocystis spp."),
    ("chilomastix", "Chilomastix mesnili"),
    ("endolimax", "Endolimax nana"),
    ("hartmanni", "Entamoeba hartmanni"),
    ("entamoeba coli", "Entamoeba coli"),
    ("iodamoeba", "Iodamoeba butschlii"),
    ("coccid", "Coccidios intestinales"),
    ("strongyloides", "Strongyloides (larva)"),
]


def norm(nombre: str) -> str:
    t = nombre.lower().replace("ü", "u").replace("ö", "o")
    return "".join(c if c.isalnum() else " " for c in t).split()


def canon(nombre: str):
    plano = " ".join(norm(nombre))
    for clave, destino in SINONIMOS:
        if clave in plano:
            return destino
    return None


def leer_nombres():
    data = yaml.safe_load(YAML.read_text())
    names = data["names"]
    if isinstance(names, dict):
        return [names[k] for k in sorted(names, key=lambda x: int(x))]
    return list(names)


def escribir_yaml(nombres):
    lineas = [f"path: {SALIDA.resolve()}", "train: images/train", "val: images/val", "names:"]
    lineas += [f'  {i}: "{n}"' for i, n in enumerate(nombres)]
    YAML.write_text("\n".join(lineas) + "\n")


def split_de(nombre: str) -> str:
    return "val" if RANDOM.random() < 0.15 else "train"


def guardar_caja(origen: Path, nombre_destino: str, clases_idx, cajas):
    """cajas: lista de (clase_id, cx, cy, w, h) ya normalizados."""
    if not cajas:
        return False
    split = split_de(nombre_destino)
    dir_i = SALIDA / "images" / split
    dir_l = SALIDA / "labels" / split
    dir_i.mkdir(parents=True, exist_ok=True)
    dir_l.mkdir(parents=True, exist_ok=True)
    destino = dir_i / nombre_destino
    if not destino.exists():
        try:
            destino.symlink_to(origen.resolve())
        except OSError:
            shutil.copy2(origen, destino)
    lineas = [f"{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}" for c, cx, cy, w, h in cajas]
    (dir_l / (Path(nombre_destino).stem + ".txt")).write_text("\n".join(lineas))
    return True


def caja_yolo(x1, y1, x2, y2, W, H):
    if x1 > x2:
        x1, x2 = x2, x1
    if y1 > y2:
        y1, y2 = y2, y1
    w, h = x2 - x1, y2 - y1
    if w < 2 or h < 2 or W < 2 or H < 2:
        return None
    cx = min(max((x1 + w / 2) / W, 0), 1)
    cy = min(max((y1 + h / 2) / H, 0), 1)
    return cx, cy, min(w / W, 1), min(h / H, 1)


# ---------- 1. microscopio USB (Zenodo) ----------
def sumar_usb():
    import csv
    zip_path = Path("/content/chula_usb.zip")
    if not zip_path.exists():
        print("Descargando set USB (94 MB)…")
        import urllib.request
        urllib.request.urlretrieve(
            "https://zenodo.org/api/records/10437395/files/Chula_parasitic_egg_detection.zip/content",
            zip_path,
        )
    raiz = Path("/content/chula_usb")
    if not any(raiz.rglob("model_label.csv")):
        print("Descomprimiendo set USB…")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(raiz)
    csv_path = next(raiz.rglob("model_label.csv"))
    indice = {}
    for p in raiz.rglob("*"):
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"} and "_" not in p.stem:
            indice.setdefault(p.name, p)
    por_imagen = {}
    with csv_path.open(newline="") as f:
        for row in csv.DictReader(f):
            nombre = Path(row["img_name"]).name
            if nombre not in indice:
                continue
            por_imagen.setdefault(nombre, []).append(row)
    clases_csv = {}
    class_csv = next(raiz.rglob("model_class.csv"))
    for linea in class_csv.read_text().splitlines():
        if not linea.strip():
            continue
        i, nombre = linea.split(",", 1)
        clases_csv[int(i)] = nombre.strip()
    nombres = leer_nombres()
    puestos = 0
    for nombre, filas in por_imagen.items():
        im = Image.open(indice[nombre])
        W, H = im.size
        cajas = []
        for row in filas:
            destino = canon(clases_csv[int(row["class_id"])])
            if destino is None:
                continue
            if destino not in nombres:
                nombres.append(destino)
            xywh = caja_yolo(float(row["xmin"]), float(row["ymin"]), float(row["xmax"]), float(row["ymax"]), W, H)
            if xywh:
                cajas.append((nombres.index(destino), *xywh))
        if guardar_caja(indice[nombre], "usb_" + nombre, nombres, cajas):
            puestos += 1
    escribir_yaml(nombres)
    print(f"USB: {puestos} fotos originales añadidas (sin los recortes aumentados).")


# ---------- 2. ParasitoBank (COCO) ----------
def encontrar_zip_parasito():
    candidatos = [
        Path(PROYECTO) / "ParasitoBank.zip",
        Path("/content/ParasitoBank.zip"),
    ]
    candidatos += [p for p in Path(PROYECTO).glob("*.zip") if "parasit" in p.name.lower()]
    for p in candidatos:
        if p.exists() and p.stat().st_size > 1_000_000 and "chula" not in p.name.lower():
            return p
    return None


def sumar_parasito():
    zpath = encontrar_zip_parasito()
    if zpath is None:
        print("ParasitoBank no está en Drive. Sáltandolo.")
        print("  Bájalo de https://data.mendeley.com/datasets/725hpzpwzf/1")
        print(f"  y guárdalo como {PROYECTO}/ParasitoBank.zip  (luego vuelve a correr esta celda).")
        return
    raiz = Path("/content/parasitobank")
    if not any(raiz.rglob("*.json")):
        print("Descomprimiendo", zpath.name, "…")
        with zipfile.ZipFile(zpath) as z:
            z.extractall(raiz)
    jsons = [p for p in raiz.rglob("*.json")]
    if not jsons:
        print("El zip no trae JSON COCO. No se usó ParasitoBank.")
        return
    import json as _json
    datos = _json.loads(max(jsons, key=lambda p: p.stat().st_size).read_text())
    id_a_nombre = {c["id"]: c["name"] for c in datos["categories"]}
    conteo = {}
    for a in datos["annotations"]:
        destino = canon(id_a_nombre.get(a["category_id"], ""))
        if destino:
            conteo[destino] = conteo.get(destino, 0) + 1
    nombres = leer_nombres()
    ya = set(nombres)
    admitidas = set()
    for destino, n in sorted(conteo.items()):
        if destino in ya or n >= MIN_NUEVA:
            admitidas.add(destino)
            if destino not in nombres:
                nombres.append(destino)
        else:
            print(f"  omitida «{destino}»: solo {n} marcas (mínimo {MIN_NUEVA}).")
    indice = {p.name: p for p in raiz.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
    anots = {}
    for a in datos["annotations"]:
        anots.setdefault(a["image_id"], []).append(a)
    puestos = 0
    for img in datos["images"]:
        origen = indice.get(Path(img["file_name"]).name)
        if origen is None:
            continue
        W, H = img.get("width"), img.get("height")
        if not W or not H:
            W, H = Image.open(origen).size
        cajas = []
        for a in anots.get(img["id"], []):
            destino = canon(id_a_nombre.get(a["category_id"], ""))
            if destino not in admitidas:
                continue
            if a.get("bbox") and len(a["bbox"]) == 4:
                x, y, w, h = a["bbox"]
            else:
                seg = a.get("segmentation") or []
                pts = [v for poli in seg if isinstance(poli, list) for v in poli]
                if len(pts) < 6:
                    continue
                xs, ys = pts[0::2], pts[1::2]
                x, y, w, h = min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys)
            xywh = caja_yolo(x, y, x + w, y + h, W, H)
            if xywh:
                cajas.append((nombres.index(destino), *xywh))
        if guardar_caja(origen, "pb_" + Path(img["file_name"]).name, nombres, cajas):
            puestos += 1
    escribir_yaml(nombres)
    print(f"ParasitoBank: {puestos} fotos. Clases ahora: {nombres}")


# ---------- 3. anotaciones propias en Drive ----------
def sumar_mias():
    base = Path(PROYECTO) / "mis_anotaciones"
    clases_txt = base / "classes.txt"
    if not clases_txt.exists():
        print("Sin carpeta mis_anotaciones/classes.txt. Nada propio que sumar.")
        return
    nombres = leer_nombres()
    propias = [ln.strip() for ln in clases_txt.read_text().splitlines() if ln.strip()]
    ids = []
    for nombre in propias:
        destino = canon(nombre) or nombre
        if destino not in nombres:
            nombres.append(destino)
        ids.append(nombres.index(destino))
    img_dir = base / "images"
    lbl_dir = base / "labels"
    puestos = 0
    for origen in img_dir.rglob("*"):
        if origen.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        txt = lbl_dir / (origen.stem + ".txt")
        if not txt.exists():
            continue
        cajas = []
        for linea in txt.read_text().splitlines():
            partes = linea.split()
            if len(partes) != 5:
                continue
            i = int(partes[0])
            if i < 0 or i >= len(ids):
                continue
            cx, cy, w, h = map(float, partes[1:])
            cajas.append((ids[i], cx, cy, w, h))
        if guardar_caja(origen, "mia_" + origen.name, nombres, cajas):
            puestos += 1
    escribir_yaml(nombres)
    print(f"Fotos propias: {puestos}.")


if not YAML.exists():
    raise SystemExit("Falta dataset_yolo. Corre antes el paso 4.")

for _nombre, _paso in (("set USB", sumar_usb), ("ParasitoBank", sumar_parasito), ("fotos propias", sumar_mias)):
    try:
        _paso()
    except Exception as _e:
        print(f"⚠️ No se pudo sumar {_nombre}: {type(_e).__name__}: {_e}")
        print("   Se continúa sin esa fuente. Puedes entrenar igual.")
print("Listo. data.yaml:")
print(YAML.read_text())

from collections import Counter
_cuenta = Counter()
for _t in (SALIDA / "labels").rglob("*.txt"):
    for _l in _t.read_text().splitlines():
        if _l.strip():
            _cuenta[int(_l.split()[0])] += 1
print("\nMarcas por clase:")
for _i, _n in enumerate(leer_nombres()):
    print(f"  {_n:<34} {_cuenta.get(_i, 0)}")


## 5. Entrenar
`yolo11n` es pequeño y rápido (bueno para celular). Si tienes tiempo, prueba `yolo11s.pt` para más precisión.

In [ ]:
from ultralytics import YOLO
modelo = YOLO('yolo11n.pt')   # parte de un modelo preentrenado
res = modelo.train(
    data='/content/dataset_yolo/data.yaml',
    epochs=25, imgsz=640, batch=16,
    project=PROYECTO, name='entrenamiento', exist_ok=True,
    # aumentos para parecerse a fotos tomadas con celular por el ocular:
    hsv_h=0.02, hsv_s=0.5, hsv_v=0.4, degrees=180, flipud=0.5, scale=0.5,
)
MEJOR = f'{res.save_dir}/weights/best.pt'
print('Modelo guardado en:', MEJOR)

### 5b. (Solo si Colab se desconectó) Reanudar el entrenamiento

In [ ]:
from ultralytics import YOLO
YOLO(f'{PROYECTO}/entrenamiento/weights/last.pt').train(resume=True)
MEJOR = f'{PROYECTO}/entrenamiento/weights/best.pt'

## 6. Validar con mAP

Corre esto **después** de entrenar. Mide el 15 % de fotos que no se usaron para aprender. No es un examen de tu microscopio si todavía no apartaste fotos propias.

- **mAP50**: el recuadro acierta la estructura y se solapa al menos a la mitad con la marca. En Chula, cerca de **0.90** es un buen modelo.
- **mAP50-95**: además exige que el recuadro ajuste bien el borde. Sale más bajo. No lo compares con el 0.90.
- Por clase, **mAP50 bajo 0.50** no se usa para nombrar esa estructura.

El informe queda en Drive: `detector_parasitos/entrenamiento/map.txt`.


In [ ]:
from pathlib import Path

from IPython.display import Image as Img, display
from ultralytics import YOLO

MEJOR = f"{PROYECTO}/entrenamiento/weights/best.pt"
DATA = "/content/dataset_yolo/data.yaml"
m = YOLO(MEJOR).val(data=DATA, split="val", imgsz=640, plots=True)

print(f"mAP50:    {m.box.map50:.3f}")
print(f"mAP50-95: {m.box.map:.3f}")
print()
nombres = [m.names[i] for i in range(len(m.names))]
print(f"{'clase':<34} {'mAP50':>8} {'mAP50-95':>10}")
filas = []
for i, nombre in enumerate(nombres):
    try:
        _p, _r, ap50, ap = m.box.class_result(i)
    except Exception:
        ap = float(m.box.maps[i]) if i < len(m.box.maps) else float("nan")
        ap50 = ap
    filas.append((nombre, ap50, ap))
    aviso = "  no reportar" if ap50 < 0.5 else ""
    print(f"{nombre:<34} {ap50:8.3f} {ap:10.3f}{aviso}")

debiles = [n for n, ap50, _ap in filas if ap50 < 0.5]
print()
if debiles:
    print("Por debajo de 0.50:", ", ".join(debiles))
else:
    print("Ninguna clase quedó bajo 0.50 en mAP50.")

informe = Path(PROYECTO) / "entrenamiento" / "map.txt"
lineas = [
    f"mAP50\t{m.box.map50:.3f}",
    f"mAP50-95\t{m.box.map:.3f}",
    "",
    "clase\tmAP50\tmAP50-95",
    *[f"{n}\t{a50:.3f}\t{a:.3f}" for n, a50, a in filas],
]
informe.parent.mkdir(parents=True, exist_ok=True)
informe.write_text("\n".join(lineas) + "\n")
print("Informe:", informe)

matriz = Path(m.save_dir) / "confusion_matrix_normalized.png"
if matriz.exists():
    display(Img(str(matriz), width=700))



## 7. Probar con tus propias fotos

In [ ]:
from google.colab import files
from IPython.display import Image as Img, display
from ultralytics import YOLO
subidas = files.upload()
for nombre in subidas:
    r = YOLO(MEJOR).predict(nombre, conf=0.25)[0]
    display(Img(data=__import__('cv2').imencode('.jpg', r.plot())[1].tobytes(), width=600))
    for c in r.boxes:
        print(r.names[int(c.cls)], f'{float(c.conf):.0%}')

## 8. Abrir la app web
Genera un enlace público temporal (72 h) que puedes abrir en tu celular y subir fotos directamente.

In [ ]:
!python app.py --modelo "$MEJOR" --compartir

## 9. Exportar para la app del celular (PWA)
Genera `best.onnx` en tu Drive: `detector_parasitos/entrenamiento/weights/best.onnx`. Ese archivo es el que se carga en la app.

In [ ]:
from ultralytics import YOLO
MEJOR = f'{PROYECTO}/entrenamiento/weights/best.pt'
print(YOLO(MEJOR).export(format='onnx', imgsz=640, opset=17, simplify=True))